<div style="border-left: 5px solid #b7791f; background-color: #fff8e1; padding: 0.8em 1em; margin: 1em 0; border-radius: 4px;">
  <strong>Warning: AI-assisted materials</strong><br><br>
  These materials were developed with assistance from AI tools. All content has been reviewed and edited by the instructor, who takes final responsibility for its accuracy, clarity, and appropriateness for the course. Students should treat these materials as instructor-reviewed course content while applying the same critical judgment they would use with any technical material. Please report any suspected errors or unclear explanations to ghunt@wm.edu.
</div>

# Random Forests and Bagging

In the previous lecture, we studied trees: classification and regression trees. A single tree is easy to interpret and can capture nonlinear structure, but it is often unstable. A small change in the training data can produce a noticeably different tree.

Random forests are built around one main idea: average many randomized trees. We'll see that this allows the variance to go down while the bias stays relatively low.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.datasets import make_moons, load_diabetes, load_breast_cancer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(657677)

## Recap: a single tree

A tree partitions the input space into leaf regions

$$
R_1,\dots,R_M.
$$

For regression, a tree has score function

$$
\hat s(x)=\sum_{m=1}^M \hat c_m \mathbf{1}\{x\in R_m\},
$$

where

$$
\hat c_m
=\frac{1}{N_m}\sum_{n:x_n\in R_m}y_n.
$$

For classification, each leaf has a vector of class proportions

$$
\hat p_m=\begin{bmatrix}
\hat p_{m1}\\
\vdots\\
\hat p_{mK}
\end{bmatrix},
\qquad
\hat p_{mk}
=\frac{1}{N_m}\sum_{n:x_n\in R_m}\mathbf{1}\{y_n=k\}.
$$

The score function is

$$
\hat s(x)
=\sum_{m=1}^M \hat p_m \mathbf{1}\{x\in R_m\}.
$$

Then

$$
\hat f(x)
=\arg\max_k \hat s_k(x).
$$

A single deep tree usually has low bias but high variance.

## Expectation and variance of averages

Before bagging, we need one basic fact: averaging reduces variance.

Let

$$
Z_1,\dots,Z_B
$$

be random variables with the same mean and variance:

$$
\mathbb{E}[Z_b]=\mu,
\qquad
\operatorname{Var}(Z_b)=\sigma^2.
$$

Define their average

$$
\bar Z_B
=\frac{1}{B}\sum_{b=1}^B Z_b.
$$

Then

$$
\mathbb{E}[\bar Z_B]
=\mu.
$$

So averaging does not change the mean.


If $Z_1,\dots,Z_B$ are uncorrelated, then

$$
\operatorname{Var}(\bar Z_B)
=\operatorname{Var}\left(\frac{1}{B}\sum_{b=1}^B Z_b\right)
=\frac{1}{B^2}\sum_{b=1}^B \operatorname{Var}(Z_b).
$$

Since each variance is $\sigma^2$,

$$
\operatorname{Var}(\bar Z_B)
=\frac{1}{B^2}B\sigma^2
=\frac{\sigma^2}{B}.
$$

So if the $Z_b$ are independent, averaging $B$ of them reduces variance by a factor of $B$.

In [ ]:
# Simulate the variance reduction from averaging independent estimators.

B_values = np.arange(1, 101)
n_reps = 5000

average_variances = []

for B in B_values:
    Z = rng.exponential(size=(n_reps, B))
    Z_bar = Z.mean(axis=1)
    average_variances.append(np.var(Z_bar, ddof=1))

plt.figure(figsize=(6, 4))
plt.plot(B_values, average_variances, label="simulated variance")
plt.plot(B_values, 1 / B_values, linestyle="--", label="$1/B$")
plt.xlabel("number of estimators $B$")
plt.ylabel(r"$\operatorname{Var}(\bar Z_B)$")
plt.title("Variance reduction from averaging independent estimators")
plt.legend()
plt.tight_layout()
plt.show()

Now, if the $Z_b$ aren't independent, we'll need to slightly modify things. Assume

$$
\operatorname{Var}(Z_b)=\sigma^2
$$

and every pair has the same correlation

$$
\operatorname{Corr}(Z_b,Z_{b'})=\rho
\qquad
b\neq b'.
$$

Then

$$
\operatorname{Cov}(Z_b,Z_{b'})=\rho\sigma^2.
$$

Therefore,

$$
\operatorname{Var}(\bar Z_B)
=\frac{1}{B^2}
\left[
B\sigma^2 + B(B-1)\rho\sigma^2
\right].
$$

Simplifying,

$$
\operatorname{Var}(\bar Z_B)
=\rho\sigma^2
+
\frac{1-\rho}{B}\sigma^2.
$$

Consequently:

- If $\rho=0$, then $\operatorname{Var}(\bar Z_B)=\sigma^2/B$.
- If $\rho=1$, then $\operatorname{Var}(\bar Z_B)=\sigma^2$.
- If $0<\rho<1$, averaging helps, but the variance cannot go below $\rho\sigma^2$.

So averaging reduces variance most when the observations not too correlated.

In [ ]:
# Variance of an average as a function of B for several correlations.

B_values = np.arange(1, 301)
rho_values = [0.0, 0.05, 0.2, 0.5, 0.9, 1]
sigma2 = 1.0

plt.figure(figsize=(7, 4))

for rho in rho_values:
    var_avg = rho * sigma2 + (1 - rho) * sigma2 / B_values
    plt.plot(B_values, var_avg, label=fr"$\rho={rho}$")

plt.xlabel("number of estimators $B$")
plt.ylabel(r"$\operatorname{Var}(\bar Z_B)$")
plt.title("Averaging helps less when estimators are correlated")
plt.legend()
plt.tight_layout()
plt.show()

## Bagging: bootstrap aggregation

Using these properties of averaging we can make regression and classifiers better by combining, in what's called an ensemble, many individual regression methods or classifiers. This is called bagging. Bagging (stands for **bootstrap aggregation**) whereby we:

1. draw many (**bootstrap**) sub-samples from the training data;
2. fit one tree to each bootstrap sample;
3. combine the resulting predictions.

A **bootstrap sample** from data of size $N$, is a sample of size $N$ that is drawn **with replacement** from the original training data. So each bootstrap sample contains some training observations multiple times and leaves some observations out. The main idea is that a boostrap sample mimics the sampling process:

- real sample: population $\to$ sample
- boostrap sample: sample $\to$ subsample

### Step 1

Let the original training data be

$$
\mathcal{D}_N = \{(x_n,y_n)\}_{n=1}^N.
$$

For $b=1,\dots,B$, draw a bootstrap sample

$$
\mathcal{D}_N^{(b)}
$$

by sampling $N$ training pairs from $\mathcal{D}_N$ with replacement.

### Step 2

Then fit a **base method** to each bootstrap sample:

$$
\hat s_b.
$$

### Step 3

Then we'll combine the various $\hat s_b$. 

#### Regression

For **regression**, each $\hat s$ gives a scalar score

$$
\hat s_b(x) \in \mathbb{R}.
$$

The bagged score function is the average:

$$
\hat s_{\text{bag}}(x)
=\frac{1}{B}\sum_{b=1}^B \hat s_b(x).
$$

Since regression uses the identity action,

$$
\hat f_{\text{bag}}(x)
=\hat s_{\text{bag}}(x).
$$

#### Classification

For classification, each tree produces a vector of class scores:

$$
\hat s_b(x)=
\begin{bmatrix}
\hat s_{b,1}(x)\\
\vdots\\
\hat s_{b,K}(x)
\end{bmatrix}.
$$

Recall that $\hat s_{b,k}(x)$ is the proportion of class $k$ observations in the leaf containing $x$.

We combine the trees by **averaging their score vectors**:

$$
\hat s_{\text{bag}}(x)
=\frac{1}{B}\sum_{b=1}^B \hat s_b(x).
$$

Then we predict the class with the largest average score:

$$
\hat f_{\text{bag}}(x)
=\arg\max_k \hat s_{\text{bag},k}(x).
$$

For example, suppose three trees give scores

$$
(0.8,\,0.2),\qquad
(0.6,\,0.4),\qquad
(0.4,\,0.6).
$$

Their average is $(0.6,\,0.4)$, so the bagged classifier predicts class 1.

Thus, for both regression and classification, bagging follows the same recipe: **average the trees' scores, then apply the usual prediction rule.**

> **A note on voting.** Averaging class scores is called *soft voting*. Another approach, *hard voting*, gives each tree one vote for its predicted class. These rules can sometimes produce different predictions. Also, the averaged scores are not necessarily *calibrated probabilities*: among observations given a class-1 score of $0.8$, the actual proportion belonging to class 1 may differ from 80%.

### Out-of-bag samples

Because a bootstrap sample is drawn with replacement, some training observations appear more than once, while others are not selected at all. The observations not selected are called **out-of-bag observations** for that sample. The tree fitted to that sample never sees them during training.

How often does an observation go unselected?

The probability that a fixed training observation is not selected in one draw is

$$
1-\frac{1}{N}.
$$

The probability that it is not selected in any of the $N$ bootstrap draws is

$$
\left(1-\frac{1}{N}\right)^N
\approx e^{-1}
\approx 0.368.
$$

For large $N$, this means that, on average, about $36.8\%$ of the original training observations are out of bag for any one tree. We'll use these observations later to estimate prediction error.

## Bagging trees

Deep trees can capture complex relationships, but their predictions are sensitive to changes in the training data. Fitting trees to different bootstrap samples and averaging their scores can make predictions more stable.

However, bootstrap sampling alone may still produce similar trees. For example, a strong predictor might be chosen near the top of almost every tree. Correlation limits how much averaging can reduce variance.

At a fixed input $x$, suppose individual tree scores have variance $\sigma^2$ and pairwise correlation $\rho$. Recalling our earlier calculation,

$$
\operatorname{Var}\left(\frac{1}{B}\sum_{b=1}^B \hat s_b(x)\right)
=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2.
$$

As $B\to\infty$,

$$
\operatorname{Var}\left(\frac{1}{B}\sum_{b=1}^B \hat s_b(x)\right)
\to\rho\sigma^2.
$$

More trees shrink the second term, but correlation leaves a limit on the benefit. For classification, this calculation applies to each class score separately.

This motivates random forests: can we make the trees less correlated so that averaging works better?

## From bagging to random forests

Random forests add a second source of randomness: **at each split, consider only a random subset of the features**.

Let $D$ be the total number of features and $M\leq D$ the number considered at each split. Draw a subset

$$
\mathcal{J}_{\text{split}}\subset\{1,\dots,D\},
\qquad |\mathcal{J}_{\text{split}}|=M,
$$

and choose the best split using only those $M$ features. A new subset is drawn at each split. This gives different predictors a chance to be used and can reduce correlation between the trees.

## Random forest algorithm

For $b=1,\dots,B$:

1. Draw a bootstrap sample $\mathcal{D}_N^{(b)}$ from the training data.
2. Grow a tree on that sample. At each split:
   - randomly select $M$ candidate features;
   - choose the best split among those features.
3. Continue growing the tree until a stopping rule is reached.

After fitting all $B$ trees, average their scores:

$$
\hat s_{\text{RF}}(x)=\frac{1}{B}\sum_{b=1}^B\hat s_b(x).
$$

For regression, use this average directly. For classification, predict the class with the largest average score.

Considering fewer features can make individual trees weaker and increase their bias. But it can also reduce correlation between their predictions, making averaging more effective at reducing variance.

**Random forests try to make $\rho$ smaller without making individual trees too weak.** The number of candidate features, $M$, controls this tradeoff.

## Example: classifying two moons

The moons dataset has two features, so we can plot the decision regions. We compare a single tree with a random forest. In the forest, each split considers one randomly selected feature ($M=1$).

We fit both models on the same training set. The plots show held-out test observations, and accuracy is measured on that test set.

In [ ]:
# Two features let us visualize the classification boundary.
X_moons, y_moons = make_moons(
    n_samples=500,
    noise=0.28,
    random_state=657677
)

X_moons_train, X_moons_test, y_moons_train, y_moons_test = train_test_split(
    X_moons, y_moons,
    test_size=0.30,
    stratify=y_moons,
    random_state=654654
)

In [ ]:
single_clf_tree = DecisionTreeClassifier(
    min_samples_leaf=2,
    random_state=654654
)

rf_moons = RandomForestClassifier(
    n_estimators=500,
    max_features=1,  # consider one of the two features at each split
    bootstrap=True,
    min_samples_leaf=2,
    random_state=657677
)

single_clf_tree.fit(X_moons_train, y_moons_train)
rf_moons.fit(X_moons_train, y_moons_train)

In [ ]:
#| echo: false
# Plot both decision regions on the same grid.
x1_min, x2_min = X_moons.min(axis=0) - 0.5
x1_max, x2_max = X_moons.max(axis=0) + 0.5
xx1, xx2 = np.meshgrid(
    np.linspace(x1_min, x1_max, 200),
    np.linspace(x2_min, x2_max, 200)
)
grid = np.column_stack([xx1.ravel(), xx2.ravel()])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
for ax, model, title in zip(
    axes,
    [single_clf_tree, rf_moons],
    ["Single classification tree", "Random forest"]
):
    pred = model.predict(grid).reshape(xx1.shape)
    ax.contourf(
        xx1, xx2, pred,
        levels=[-0.5, 0.5, 1.5], cmap="coolwarm", alpha=0.25
    )
    ax.scatter(
        X_moons_test[:, 0], X_moons_test[:, 1], c=y_moons_test,
        cmap="coolwarm", s=25, edgecolor="black", linewidth=0.3
    )
    accuracy = model.score(X_moons_test, y_moons_test)
    ax.set_title(f"{title}\nTest accuracy: {accuracy:.3f}")
    ax.set_xlabel("$x_1$")
axes[0].set_ylabel("$x_2$")
plt.tight_layout()
plt.show()

Both models use axis-aligned splits and achieve similar test accuracy on this split. The forest combines class scores from many randomized trees before choosing a class, so its prediction does not depend on a single fitted tree.

## Out-of-bag error

We can use the observations each tree did not see to evaluate the forest. For tree $b$, let

$$
O_b
$$

be the set of training indices that were **out of bag** for that tree.

For a fixed training point $x_n$, define

$$
B_n
=\{b: n\in O_b\},
$$

the set of trees for which observation $n$ was out of bag.

The out-of-bag score for $x_n$ is

$$
\hat s_{\text{OOB},n}(x_n)
=\frac{1}{|B_n|}\sum_{b\in B_n}\hat s_b(x_n).
$$

This is a prediction for $x_n$ **made only by trees that did not train on $x_n$**. So we can define

$$
\hat{y}_{\text{OOB},n} = a(\hat s_{\text{OOB},n}(x_n))
$$
and then use these OOB predictions to calculate error metrics. 

OOB error acts like an internal validation error. It is not exactly the same as a separate test set, but it is often a useful estimate of generalization performance since we're predicting on $x_n$ using a method that never saw $x_n$. 

In [ ]:
# Refit the same forest with OOB scoring enabled.
rf_moons.set_params(oob_score=True)
rf_moons.fit(X_moons_train, y_moons_train)

print("Training accuracy:", round(rf_moons.score(X_moons_train, y_moons_train), 3))
print("OOB accuracy:", round(rf_moons.oob_score_, 3))
print("Test accuracy:", round(rf_moons.score(X_moons_test, y_moons_test), 3))

## How many trees?

The number of trees is controlled by

$$
B.
$$

As $B$ increases, the random forest prediction stabilizes. More trees do not cause overfitting in the same way deeper individual trees can. Instead, more trees reduce Monte Carlo noise from the randomization.

The tradeoff is computation: more trees take more time and memory.

In [ ]:
# OOB and test misclassification rates for the same moons example.
n_tree_values = [1,2,5,10, 25, 50, 100, 200, 400]
oob_error_values = []
test_error_values = []

for B in n_tree_values:
    rf = RandomForestClassifier(
        n_estimators=B,
        max_features=1,
        bootstrap=True,
        oob_score=True,
        min_samples_leaf=2,
        random_state=657677
    )
    rf.fit(X_moons_train, y_moons_train)
    oob_error_values.append(1 - rf.oob_score_)
    test_error_values.append(1 - rf.score(X_moons_test, y_moons_test))

plt.figure(figsize=(7, 4))
plt.plot(n_tree_values, oob_error_values, marker="o", label="OOB error")
plt.plot(n_tree_values, test_error_values, marker="o", label="test error")
plt.xlabel("number of trees")
plt.ylabel("misclassification rate")
plt.title("OOB and test error usually stabilize as the forest grows")
plt.legend()
plt.tight_layout()
plt.show()

## Real regression example: diabetes data

We now compare a single tree, bagged trees, and a random forest on a real regression dataset.

The response is a quantitative disease-progression outcome. The predictors are baseline measurements.

In [ ]:
# Load diabetes data.

diabetes = load_diabetes(as_frame=True)
X_diab = diabetes.data
y_diab = diabetes.target

X_train, X_test, y_train, y_test = train_test_split(
    X_diab,
    y_diab,
    test_size=0.30,
    random_state=657677
)

models_reg = {
    "single tree": DecisionTreeRegressor(
        min_samples_leaf=5,
        random_state=657677
    ),
    "bagged trees": RandomForestRegressor(
        n_estimators=500,
        max_features=1.0,
        bootstrap=True,
        min_samples_leaf=5,
        oob_score=True,
        random_state=657677
    ),
    "random forest": RandomForestRegressor(
        n_estimators=500,
        max_features="sqrt",
        bootstrap=True,
        min_samples_leaf=5,
        oob_score=True,
        random_state=657677
    ),
}

rows = []

for name, model in models_reg.items():
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    row = {
        "model": name,
        "train_RMSE": np.sqrt(mean_squared_error(y_train, pred_train)),
        "test_RMSE": np.sqrt(mean_squared_error(y_test, pred_test)),
    }

    if hasattr(model, "oob_prediction_"):
        row["OOB_RMSE"] = np.sqrt(mean_squared_error(y_train, model.oob_prediction_))
    else:
        row["OOB_RMSE"] = np.nan

    rows.append(row)

results_reg = pd.DataFrame(rows)
display(results_reg.round(3))

In [ ]:
# Feature importance for the random forest regression model.

rf_reg = models_reg["random forest"]

importance_reg = pd.DataFrame({
    "feature": X_diab.columns,
    "impurity_importance": rf_reg.feature_importances_,
}).sort_values("impurity_importance", ascending=False)

display(importance_reg.round(4))

plt.figure(figsize=(7, 4))
plt.barh(importance_reg["feature"], importance_reg["impurity_importance"])
plt.gca().invert_yaxis()
plt.xlabel("impurity-based importance")
plt.title("Random forest feature importance, diabetes regression")
plt.tight_layout()
plt.show()

## Feature importance

Random forests often report feature importance scores. The usual impurity-based score works like this:

- every time a feature is used in a split, measure how much the split reduces RSS or impurity;
- add those reductions across all splits in all trees;
- normalize so that the scores sum to one.

These scores are useful, but they should be interpreted cautiously. They can be biased toward variables with many possible split points, and correlated features can split importance between each other.

A more model-agnostic alternative is **permutation importance**: randomly permute one feature in the test set and measure how much performance gets worse.

In [ ]:
# Permutation importance on the test set.

perm_reg = permutation_importance(
    rf_reg,
    X_test,
    y_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=15,
    random_state=657677
)

perm_importance_reg = pd.DataFrame({
    "feature": X_diab.columns,
    "permutation_importance": perm_reg.importances_mean,
    "sd": perm_reg.importances_std,
}).sort_values("permutation_importance", ascending=False)

display(perm_importance_reg.round(4))

## Real classification example: breast cancer data

Now we use a binary classification dataset. We compare a single classification tree, bagged trees, and a random forest.

In [ ]:
# Load breast cancer data.

cancer = load_breast_cancer(as_frame=True)
X_cancer = cancer.data
y_cancer = cancer.target
class_names = cancer.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X_cancer,
    y_cancer,
    test_size=0.30,
    stratify=y_cancer,
    random_state=657677
)

models_clf = {
    "single tree": DecisionTreeClassifier(
        min_samples_leaf=5,
        random_state=657677
    ),
    "bagged trees": RandomForestClassifier(
        n_estimators=500,
        max_features=1.0,
        bootstrap=True,
        min_samples_leaf=5,
        oob_score=True,
        random_state=657677
    ),
    "random forest": RandomForestClassifier(
        n_estimators=500,
        max_features="sqrt",
        bootstrap=True,
        min_samples_leaf=5,
        oob_score=True,
        random_state=657677
    ),
}

rows = []

for name, model in models_clf.items():
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    row = {
        "model": name,
        "train_accuracy": accuracy_score(y_train, pred_train),
        "test_accuracy": accuracy_score(y_test, pred_test),
    }

    if hasattr(model, "oob_score_"):
        row["OOB_accuracy"] = model.oob_score_
    else:
        row["OOB_accuracy"] = np.nan

    rows.append(row)

results_clf = pd.DataFrame(rows)
display(results_clf.round(4))

In [ ]:
# Confusion matrix for the random forest classifier.

rf_clf = models_clf["random forest"]
y_pred = rf_clf.predict(X_test)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=class_names
)
plt.title("Random forest confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance for the classification forest.

importance_clf = pd.DataFrame({
    "feature": X_cancer.columns,
    "impurity_importance": rf_clf.feature_importances_,
}).sort_values("impurity_importance", ascending=False)

display(importance_clf.head(15).round(4))

plt.figure(figsize=(7, 5))
plt.barh(importance_clf.head(15)["feature"], importance_clf.head(15)["impurity_importance"])
plt.gca().invert_yaxis()
plt.xlabel("impurity-based importance")
plt.title("Top random forest feature importances, breast cancer data")
plt.tight_layout()
plt.show()

## Main tuning parameters

Important random forest tuning parameters include:

**Number of trees**

$$
B.
$$

More trees reduce Monte Carlo noise, but increase computation.

**Number of features considered at each split**

$$
M.
$$

Smaller $M$ makes trees less correlated, but each individual split may be weaker.

**Tree size**

Controlled through parameters such as maximum depth, minimum samples per leaf, or minimum samples needed to split. Deeper trees have lower bias but higher variance before averaging.

## Practical notes

### Scaling

Random forests are usually not sensitive to monotone rescaling of individual features. A split like

$$
x_j \leq t
$$

has an equivalent split after changing units. Standardization is therefore not usually required.

### Categorical variables

Conceptually, trees can split categorical variables by grouping categories. Implementation details vary. In many Python workflows, categorical variables are one-hot encoded before using `sklearn` random forests.

### Missing values

Some modern tree implementations handle missing values internally. Depending on the software and version, `sklearn` support varies by tree estimator and criterion. A safe general workflow is to use an imputer inside a pipeline when missing values are present.

### Interpretation

A single tree can be drawn and inspected. A random forest is harder to interpret because it averages many trees. Feature importance and partial dependence plots can help summarize the fitted forest, but they are not the same as reading off a simple set of rules.

### Extrapolation

Trees and random forests are piecewise constant or averages of piecewise constant functions. They are usually poor at extrapolating outside the range of the training data.

## Review Questions

See: @sec-rf-questions.